# Incident Response Runbook: Lateral Movement - Remote Services

**Tactic:** Lateral Movement
**Technique:** Remote Services (T1021)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for Remote Services activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add the project root to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()

# Query Splunk for remote services lateral movement indicators
print(f"\n[QUERY] Searching Splunk for remote services lateral movement indicators...")
splunk_query = '''
index=* (sourcetype=WinEventLog:Security EventCode=4624 OR EventCode=4625 OR EventCode=4648)
OR (sourcetype=linux_secure "sshd" OR "ssh" OR "rdp" OR "vnc" OR "winrm")
OR (sourcetype=network "SMB" OR "RDP" OR "SSH" OR "VNC" OR "WinRM")
| eval service_type=case(match(LogonType, "3|10") OR match(service, "rdp|terminal"), "RDP",
                        match(service, "ssh|sshd"), "SSH",
                        match(service, "winrm|wsman"), "WinRM",
                        match(service, "vnc"), "VNC",
                        match(service, "smb|cifs"), "SMB", "unknown")
| where service_type != "unknown"
| stats count by host, user, source_host, service_type, _time
| where count > 3
'''

try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-24h")
    print(f"   Found {len(splunk_results)} potential remote services lateral movement events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

# Extract affected systems and suspicious activities
affected_systems = []
suspicious_activities = []
splunk_indicators = []
unique_users = set()
source_hosts = set()

for event in splunk_results:
    system_info = {
        'hostname': event.get('host', 'unknown'),
        'user': event.get('user', 'unknown'),
        'source_host': event.get('source_host', 'unknown'),
        'service_type': event.get('service_type', 'unknown'),
        'event_count': event.get('count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)
    unique_users.add(event.get('user', 'unknown'))
    source_hosts.add(event.get('source_host', 'unknown'))

    # Extract indicators
    if event.get('source_host') and event.get('source_host') != 'unknown':
        splunk_indicators.append({
            'type': 'remote_service_access',
            'value': f"{event.get('source_host')} -> {event.get('host')} ({event.get('service_type')})",
            'context': f"User {event.get('user', 'unknown')} accessed {event.get('host')} via {event.get('service_type')}"
        })

# Query CrowdStrike for endpoint detection
print(f"\n[QUERY] Checking CrowdStrike for remote services lateral movement detections...")
try:
    cs_detections = crowdstrike.get_detections(
        filter="technique:'Remote Services'",
        start_time="-24h"
    )
    cs_affected_hosts = []
    for detection in cs_detections:
        host_info = {
            'device_id': detection.get('device_id', ''),
            'hostname': detection.get('hostname', ''),
            'detection_id': detection.get('detection_id', ''),
            'technique': detection.get('technique', ''),
            'severity': detection.get('severity', 0)
        }
        cs_affected_hosts.append(host_info)

        # Merge with Splunk data
        existing_system = next((s for s in affected_systems if s['hostname'] == host_info['hostname']), None)
        if existing_system:
            existing_system.update(host_info)
        else:
            affected_systems.append(host_info)

    print(f"   Found {len(cs_affected_hosts)} CrowdStrike detections")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")
    cs_affected_hosts = []

# Enrich with threat intelligence from MISP
print(f"\n[ENRICHMENT] Checking MISP for related threat intelligence...")
misp_results = []
try:
    for indicator in splunk_indicators:
        # Extract IP addresses from source hosts
        ip_match = re.search(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', indicator['value'])
        if ip_match:
            misp_search = misp.search_iocs(ip_match.group())
            if misp_search:
                misp_results.extend(misp_search)
                indicator['threat_intel'] = misp_search[0] if misp_search else None
                print(f"   Found threat intelligence for {ip_match.group()}")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'Remote Services Lateral Movement Incident - {len(affected_systems)} systems',
        'description': f'Detected remote services lateral movement activities on {len(affected_systems)} systems',
        'severity': 'HIGH',
        'tactic': 'Lateral Movement',
        'technique': 'Remote Services (T1021)',
        'indicators': splunk_indicators,
        'affected_systems': affected_systems,
        'threat_intelligence': misp_results
    }

    incident_id = iris.create_case(incident_data)
    if incident_id:
        print(f"   Created IRIS case: {incident_id}")
    else:
        print(f"   Failed to create IRIS case")
        incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"LOCAL-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n Detection complete:")
print(f"  - Affected systems: {len(affected_systems)}")
print(f"  - Unique users: {len(unique_users)}")
print(f"  - Source hosts: {len(source_hosts)}")
print(f"  - Suspicious indicators: {len(splunk_indicators)}")
print(f"  - Threat intelligence hits: {len(misp_results)}")
print(f"  - Incident ID: {incident_id}")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
containment_actions = []
isolated_hosts = []
disabled_accounts = []
blocked_ips = []

# 1. Isolate affected systems
print(f"\n[CONTAINMENT] Isolating affected systems...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Use CrowdStrike to isolate host
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result:
                isolated_hosts.append(system['hostname'])
                containment_actions.append({
                    'action': 'host_isolation',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Isolated host: {system['hostname']}")
            else:
                print(f"   Failed to isolate host: {system['hostname']}")
        else:
            # Use Shuffle for network isolation
            network_isolation = shuffle.isolate_system(system['hostname'])
            if network_isolation:
                isolated_hosts.append(system['hostname'])
                containment_actions.append({
                    'action': 'network_isolation',
                    'target': system['hostname'],
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Network isolated: {system['hostname']}")
except Exception as e:
    print(f"   Error during host isolation: {e}")

# 2. Disable suspicious accounts
print(f"\n[CONTAINMENT] Disabling suspicious accounts...")
try:
    for user in unique_users:
        if user and user != 'unknown':
            # Disable account via Shuffle
            disable_result = shuffle.disable_account(user)
            if disable_result:
                disabled_accounts.append(user)
                containment_actions.append({
                    'action': 'account_disable',
                    'target': user,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Disabled account: {user}")
except Exception as e:
    print(f"   Error disabling accounts: {e}")

# 3. Block suspicious source IPs
print(f"\n[CONTAINMENT] Blocking suspicious source IPs...")
try:
    for host in source_hosts:
        # Extract IP addresses from source hosts
        ip_match = re.search(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', host)
        if ip_match:
            ip = ip_match.group()
            block_result = shuffle.block_ip(ip)
            if block_result:
                blocked_ips.append(ip)
                containment_actions.append({
                    'action': 'ip_block',
                    'target': ip,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Blocked source IP: {ip}")
except Exception as e:
    print(f"   Error blocking IPs: {e}")

# 4. Disable remote service access
print(f"\n[CONTAINMENT] Disabling remote service access...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Disable remote services on affected systems
            service_disable = crowdstrike.disable_remote_services(system['device_id'])
            if service_disable:
                containment_actions.append({
                    'action': 'remote_services_disable',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': containment_time
                })
                print(f"   Disabled remote services on: {system['hostname']}")
except Exception as e:
    print(f"   Error disabling remote services: {e}")

# 5. Enable enhanced monitoring
print(f"\n[CONTAINMENT] Enabling enhanced monitoring...")
try:
    monitoring_rules = [
        {
            'name': 'Enhanced Remote Services Monitoring',
            'query': 'index=* (EventCode=4624 OR EventCode=4625) LogonType IN (3,10) | eval risk_score = case(match(source_host, "external|unknown"), 9, match(user, "admin|administrator|root"), 8, 1==1, 6) | where risk_score >= 7',
            'alert_threshold': 2,
            'time_window': '5m'
        }
    ]

    enhanced_monitoring = splunk.enable_enhanced_monitoring(monitoring_rules)
    if enhanced_monitoring:
        containment_actions.append({
            'action': 'enhanced_monitoring',
            'target': 'Splunk',
            'method': 'Splunk',
            'status': 'success',
            'timestamp': containment_time
        })
        print(f"   Enabled enhanced monitoring in Splunk")
except Exception as e:
    print(f"   Error enabling enhanced monitoring: {e}")

print(f"\n Containment complete:")
print(f"  - Hosts isolated: {len(isolated_hosts)}")
print(f"  - Accounts disabled: {len(disabled_accounts)}")
print(f"  - IPs blocked: {len(blocked_ips)}")
print(f"  - Remote services disabled: ")
print(f"  - Enhanced monitoring: enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

eradication_time = datetime.now().isoformat()
eradication_actions = []
terminated_processes = []
deleted_tools = []
reset_credentials = []

# 1. Terminate suspicious remote sessions
print(f"\n[ERADICATION] Terminating suspicious remote sessions...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Get active remote sessions from CrowdStrike
            sessions = crowdstrike.get_remote_sessions(system['device_id'])
            suspicious_sessions = []

            for session in sessions:
                # Check if session is from suspicious source
                if session.get('source_host') in source_hosts or session.get('user') in unique_users:
                    suspicious_sessions.append(session)

            for session in suspicious_sessions:
                kill_result = crowdstrike.terminate_session(system['device_id'], session['session_id'])
                if kill_result:
                    terminated_processes.append(f"{system['hostname']}:{session.get('process_name', 'remote_session')}")
                    eradication_actions.append({
                        'action': 'session_termination',
                        'target': f"{system['hostname']}:{session.get('process_name', 'remote_session')}",
                        'method': 'CrowdStrike',
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Terminated remote session: {session.get('process_name', 'remote_session')} on {system['hostname']}")
except Exception as e:
    print(f"   Error terminating sessions: {e}")

# 2. Remove lateral movement tools
print(f"\n[ERADICATION] Removing lateral movement tools...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Scan for and remove common lateral movement tools
            tools_to_remove = [
                'psexec.exe', 'wmic.exe', 'powershell.exe', 'ssh.exe',
                'mstsc.exe', 'rdpclip.exe', 'lateral.ps1', 'move.bat'
            ]

            for tool in tools_to_remove:
                removal_result = crowdstrike.remove_file(system['device_id'], tool)
                if removal_result:
                    deleted_tools.append(f"{system['hostname']}:{tool}")
                    eradication_actions.append({
                        'action': 'tool_removal',
                        'target': f"{system['hostname']}:{tool}",
                        'method': 'CrowdStrike',
                        'status': 'success',
                        'timestamp': eradication_time
                    })
                    print(f"   Removed tool: {tool} from {system['hostname']}")
except Exception as e:
    print(f"   Error removing tools: {e}")

# 3. Reset compromised credentials
print(f"\n[ERADICATION] Resetting compromised credentials...")
try:
    for user in unique_users:
        if user and user != 'unknown':
            # Reset password via Shuffle
            reset_result = shuffle.reset_password(user)
            if reset_result:
                reset_credentials.append(user)
                eradication_actions.append({
                    'action': 'credential_reset',
                    'target': user,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Reset credentials for: {user}")
except Exception as e:
    print(f"   Error resetting credentials: {e}")

# 4. Clean up remote service configurations
print(f"\n[ERADICATION] Cleaning up remote service configurations...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Remove unauthorized remote service configurations
            config_cleanup = crowdstrike.cleanup_remote_service_configs(system['device_id'])
            if config_cleanup:
                eradication_actions.append({
                    'action': 'config_cleanup',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': eradication_time
                })
                print(f"   Cleaned remote service configs on: {system['hostname']}")
except Exception as e:
    print(f"   Error cleaning configurations: {e}")

# 5. Verify eradication
print(f"\n[ERADICATION] Verifying eradication...")
try:
    verification_results = []
    for system in affected_systems:
        if system.get('device_id'):
            # Run verification scan for lateral movement indicators
            scan_result = crowdstrike.scan_system(system['device_id'], 'lateral_movement_indicators')
            verification_results.append({
                'system': system['hostname'],
                'scan_result': scan_result,
                'clean': scan_result.get('threats_found', 0) == 0
            })

    clean_systems = [r for r in verification_results if r['clean']]
    eradication_actions.append({
        'action': 'eradication_verification',
        'target': f"{len(clean_systems)}/{len(verification_results)} systems clean",
        'method': 'CrowdStrike',
        'status': 'success' if len(clean_systems) == len(verification_results) else 'partial',
        'timestamp': eradication_time
    })

    print(f"   Verification complete: {len(clean_systems)}/{len(verification_results)} systems clean")

except Exception as e:
    print(f"   Error during verification: {e}")

print(f"\n Eradication complete:")
print(f"  - Sessions terminated: {len(terminated_processes)}")
print(f"  - Tools deleted: {len(deleted_tools)}")
print(f"  - Credentials reset: {len(reset_credentials)}")
print(f"  - Systems verified clean: {len([r for r in verification_results if r.get('clean', False)])}")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

recovery_time = datetime.now().isoformat()
recovery_actions = []
reenabled_hosts = []
restored_accounts = []
restored_services = []

# 1. Re-enable isolated systems
print(f"\n[RECOVERY] Re-enabling isolated systems...")
try:
    for host in isolated_hosts:
        system = next((s for s in affected_systems if s['hostname'] == host), None)
        if system and system.get('device_id'):
            # Use CrowdStrike to re-enable host
            reenable_result = crowdstrike.reenable_host(system['device_id'])
            if reenable_result:
                reenabled_hosts.append(host)
                recovery_actions.append({
                    'action': 'host_reenable',
                    'target': host,
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Re-enabled host: {host}")
            else:
                print(f"   Failed to re-enable host: {host}")
        else:
            # Use Shuffle for network re-enablement
            network_restore = shuffle.restore_system(host)
            if network_restore:
                reenabled_hosts.append(host)
                recovery_actions.append({
                    'action': 'network_restore',
                    'target': host,
                    'method': 'Shuffle',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Restored network access: {host}")
except Exception as e:
    print(f"   Error re-enabling systems: {e}")

# 2. Restore disabled accounts
print(f"\n[RECOVERY] Restoring disabled accounts...")
try:
    for user in disabled_accounts:
        # Restore account access via Shuffle
        restore_result = shuffle.restore_account(user)
        if restore_result:
            restored_accounts.append(user)
            recovery_actions.append({
                'action': 'account_restore',
                'target': user,
                'method': 'Shuffle',
                'status': 'success',
                'timestamp': recovery_time
            })
            print(f"   Restored account: {user}")
except Exception as e:
    print(f"   Error restoring accounts: {e}")

# 3. Restore remote services securely
print(f"\n[RECOVERY] Restoring remote services securely...")
try:
    for system in affected_systems:
        if system.get('device_id'):
            # Restore remote services with enhanced security
            service_restore = crowdstrike.restore_remote_services_secure(system['device_id'])
            if service_restore:
                restored_services.append(system['hostname'])
                recovery_actions.append({
                    'action': 'service_restore',
                    'target': system['hostname'],
                    'method': 'CrowdStrike',
                    'status': 'success',
                    'timestamp': recovery_time
                })
                print(f"   Restored remote services securely on: {system['hostname']}")
except Exception as e:
    print(f"   Error restoring services: {e}")

# 4. Restore monitoring and alerting
print(f"\n[RECOVERY] Restoring monitoring and alerting...")
try:
    # Restore normal Splunk monitoring rules
    normal_monitoring = splunk.restore_normal_monitoring()
    if normal_monitoring:
        recovery_actions.append({
            'action': 'monitoring_restore',
            'target': 'Splunk',
            'method': 'Splunk',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal monitoring in Splunk")

    # Restore CrowdStrike normal operations
    cs_normal_ops = crowdstrike.restore_normal_operations()
    if cs_normal_ops:
        recovery_actions.append({
            'action': 'cs_normal_ops',
            'target': 'CrowdStrike',
            'method': 'CrowdStrike',
            'status': 'success',
            'timestamp': recovery_time
        })
        print(f"   Restored normal CrowdStrike operations")

except Exception as e:
    print(f"   Error restoring monitoring: {e}")

# 5. Validate recovery
print(f"\n[RECOVERY] Validating recovery...")
try:
    validation_results = []
    for system in affected_systems:
        # Test remote service functionality securely
        test_result = shuffle.test_remote_service_functionality(system['hostname'])
        validation_results.append({
            'system': system['hostname'],
            'functional': test_result.get('success', False),
            'services': test_result.get('services_running', [])
        })

    functional_systems = [r for r in validation_results if r['functional']]
    recovery_actions.append({
        'action': 'recovery_validation',
        'target': f"{len(functional_systems)}/{len(validation_results)} systems functional",
        'method': 'Shuffle',
        'status': 'success' if len(functional_systems) == len(validation_results) else 'partial',
        'timestamp': recovery_time
    })

    print(f"   Validation complete: {len(functional_systems)}/{len(validation_results)} systems functional")

except Exception as e:
    print(f"   Error during validation: {e}")

print(f"\n Recovery complete:")
print(f"  - Hosts re-enabled: {len(reenabled_hosts)}")
print(f"  - Accounts restored: {len(restored_accounts)}")
print(f"  - Services restored: {len(restored_services)}")
print(f"  - Systems validated: {len([r for r in validation_results if r.get('functional', False)])}")
print(f"  - Monitoring restored: ")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []
closure_time = datetime.now().isoformat()

# 1. Generate comprehensive incident report
print(f"\n[POST-INCIDENT] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'title': f'Remote Services Lateral Movement Incident Report - {len(affected_systems)} systems',
        'detection_time': detection_time,
        'closure_time': closure_time,
        'severity': 'HIGH',
        'tactic': 'Lateral Movement',
        'technique': 'Remote Services (T1021)',
        'summary': {
            'affected_users': len(unique_users),
            'affected_systems': len(affected_systems),
            'source_hosts': len(source_hosts),
            'indicators_detected': len(splunk_indicators),
            'hosts_isolated': len(isolated_hosts),
            'accounts_disabled': len(disabled_accounts),
            'processes_terminated': len(terminated_processes),
            'tools_deleted': len(deleted_tools),
            'credentials_reset': len(reset_credentials)
        },
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': eradication_time,
            'recovery': recovery_time,
            'closure': closure_time
        },
        'actions_taken': {
            'detection': splunk_indicators[:10],  # Top 10 indicators
            'containment': containment_actions,
            'eradication': eradication_actions,
            'recovery': recovery_actions
        },
        'threat_intelligence': [i.get('threat_intel') for i in splunk_indicators if i.get('threat_intel')],
        'recommendations': [
            'Implement remote service access monitoring and alerting',
            'Restrict remote access to authorized systems only',
            'Deploy multi-factor authentication for remote services',
            'Regular security awareness training on lateral movement detection',
            'Enhance logging for remote service authentication and access'
        ]
    }

    # Save report to file
    report_filename = f"remote_services_lateral_movement_report_{incident_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(report_filename, 'w') as f:
        json.dump(incident_report, f, indent=2, default=str)

    post_incident_actions.append({
        'action': 'report_generation',
        'target': report_filename,
        'status': 'success',
        'timestamp': closure_time
    })
    print(f"   Generated incident report: {report_filename}")

except Exception as e:
    print(f"   Error generating report: {e}")

# 2. Document lessons learned
print(f"\n[POST-INCIDENT] Documenting lessons learned...")
try:
    lessons_learned = {
        'incident_id': incident_id,
        'what_went_well': [
            'Rapid detection of remote service lateral movement',
            'Effective isolation of affected systems',
            'Comprehensive eradication of lateral movement tools',
            'Successful system recovery and validation'
        ],
        'what_could_improve': [
            'Earlier detection of remote service abuse patterns',
            'Enhanced monitoring of remote authentication',
            'Automated blocking of suspicious remote access',
            'Better user training on secure remote access practices'
        ],
        'key_findings': [
            f'Lateral movement affected {len(unique_users)} users across {len(affected_systems)} systems from {len(source_hosts)} sources',
            f'Most common service abused: {max([s.get("service_type", "unknown") for s in affected_systems], key=[s.get("service_type", "unknown") for s in affected_systems].count)}',
            'Threat intelligence enriched detection for key lateral movement indicators',
            'Automated response prevented further network compromise'
        ],
        'preventive_measures': [
            'Implement just-in-time access for remote services',
            'Deploy advanced endpoint protection with lateral movement detection',
            'Regular vulnerability scanning and remote service hardening',
            'Enhanced security monitoring and alerting for remote access',
            'Security awareness training focused on lateral movement prevention'
        ]
    }

    # Save lessons learned
    lessons_filename = f"remote_services_lateral_movement_lessons_{incident_id}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(lessons_filename, 'w') as f:
        json.dump(lessons_learned, f, indent=2, default=str)

    post_incident_actions.append({
        'action': 'lessons_learned',
        'target': lessons_filename,
        'status': 'success',
        'timestamp': closure_time
    })
    print(f"   Documented lessons learned: {lessons_filename}")

except Exception as e:
    print(f"   Error documenting lessons learned: {e}")

# 3. Implement preventive measures
print(f"\n[POST-INCIDENT] Implementing preventive measures...")
try:
    preventive_measures = []

    # Update Splunk correlation rules for lateral movement
    updated_rules = splunk.update_correlation_rules([
        {
            'name': 'Enhanced Remote Services Lateral Movement Detection',
            'search': 'index=* (EventCode=4624 OR EventCode=4625) LogonType IN (3,10) | eval risk_score = case(match(source_host, "external|unknown"), 9, match(user, "admin|administrator|root"), 8, match(service, "rdp|ssh|winrm"), 7, 1==1, 5) | where risk_score >= 7',
            'alert_threshold': 2,
            'time_window': '5m'
        }
    ])
    if updated_rules:
        preventive_measures.append('Updated Splunk lateral movement rules')
        print(f"   Updated Splunk correlation rules for lateral movement detection")

    # Enhance CrowdStrike prevention policies
    cs_policies = crowdstrike.update_prevention_policies({
        'lateral_movement_prevention': 'enabled',
        'remote_service_monitoring': 'strict',
        'suspicious_remote_access_detection': 'enabled',
        'credential_theft_protection': 'enabled'
    })
    if cs_policies:
        preventive_measures.append('Enhanced CrowdStrike lateral movement policies')
        print(f"   Enhanced CrowdStrike prevention policies")

    # Schedule lateral movement security training
    training_scheduled = shuffle.schedule_security_training(
        users=list(unique_users),
        topic='Detecting and Preventing Lateral Movement via Remote Services',
        incident_reference=incident_id
    )
    if training_scheduled:
        preventive_measures.append('Scheduled lateral movement training')
        print(f"   Scheduled security awareness training on lateral movement detection")

except Exception as e:
    print(f"   Error implementing preventive measures: {e}")

# 4. Share threat intelligence
print(f"\n[POST-INCIDENT] Sharing threat intelligence...")
try:
    shared_intel = []
    for indicator in splunk_indicators:
        if indicator.get('threat_intel'):
            # Share with MISP
            result = misp.share_indicator(indicator, incident_id)
            if result:
                shared_intel.append(f"MISP: {indicator.get('type', 'unknown')}")
                print(f"   Shared indicator with MISP: {indicator.get('type', 'unknown')}")

    if shared_intel:
        post_incident_actions.append({
            'action': 'threat_intel_sharing',
            'target': shared_intel,
            'status': 'success',
            'timestamp': closure_time
        })

except Exception as e:
    print(f"   Error sharing threat intelligence: {e}")

# 5. Close incident case
print(f"\n[POST-INCIDENT] Closing incident case...")
try:
    if incident_id:
        closure_data = {
            'status': 'closed',
            'closure_time': closure_time,
            'closure_reason': 'Remote services lateral movement incident successfully contained, eradicated, and recovered',
            'post_incident_actions': post_incident_actions,
            'final_assessment': {
                'threat_contained': len(isolated_hosts) > 0,
                'threat_eradicated': len(terminated_processes) > 0 or len(deleted_tools) > 0,
                'systems_recovered': len(reenabled_hosts) > 0,
                'preventive_measures': len(preventive_measures) > 0
            }
        }

        result = iris.close_case(incident_id, closure_data)
        if result:
            post_incident_actions.append({
                'action': 'case_closure',
                'target': incident_id,
                'status': 'success',
                'timestamp': closure_time
            })
            print(f"   Closed IRIS case: {incident_id}")
        else:
            print(f"   Failed to close IRIS case: {incident_id}")

except Exception as e:
    print(f"   Error closing incident case: {e}")

print(f"\n Post-incident activities complete:")
print(f"  - Incident report generated")
print(f"  - Lessons learned documented")
print(f"  - {len(preventive_measures)} preventive measures implemented")
print(f"  - Threat intelligence shared with {len(shared_intel)} platforms")
print(f"  - Incident case closed: {incident_id}")

print(f"\n Remote Services Lateral Movement Incident Response Complete")
print(f"   Total duration: {(datetime.fromisoformat(closure_time) - datetime.fromisoformat(detection_time)).total_seconds() / 3600:.1f} hours")
print(f"   Response effectiveness: {'HIGH' if len(isolated_hosts) > 0 and len(terminated_processes) > 0 else 'MEDIUM'}")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
